#### Imports

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from networkx.algorithms import bipartite
import random
import ast

#### Load graph data

In [ ]:
G = nx.Graph(name="Steam")

with open('../data/australian_users_items.json', encoding='utf-8') as f:
    for line in f:
        user = ast.literal_eval(line)

        user_id = user['steam_id']
        user_node = ('user', user_id)
        G.add_node(user_node, bipartite=0, node_type='user')

        for item in user['items']:
            item_id = item['item_id']
            item_name = item['item_name']

            item_node = ('item', item_id)
            G.add_node(item_node, bipartite=1, node_type='item', name=item_name)
            G.add_edge(user_node, item_node)

### Basic Graph Statistics

We compute the number of nodes, edges, density, and connected components to understand the overall structure of the network.

These metrics provide insight into:
- **Scale** of the dataset (number of users and items)
- **Sparsity** (interaction density is typically very low in recommendation systems)
- **Connectivity**, which determines how effectively information can propagate in graph-based models

A large, sparse graph with a dominant connected component is typical for real-world recommendation systems.

In [ ]:
def basic_stats(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()

    users = [n for n, d in G.nodes(data=True) if d['node_type'] == 'user']
    items = [n for n, d in G.nodes(data=True) if d['node_type'] == 'item']

    print(f"Nodes: {n}")
    print(f"Edges: {m}")
    print(f"Users: {len(users)}")
    print(f"Items: {len(items)}")

    density = nx.density(G)
    print(f"Density: {density:.6f}")

    components = list(nx.connected_components(G))
    print(f"Connected components: {len(components)}")

    largest_cc = max(components, key=len)
    print(f"Largest CC size: {len(largest_cc)}")


basic_stats(G)

### Degree Distribution

We analyze the degree distribution separately for users and items.

This is important because:
- **Item degree** reflects popularity (how many users own a game)
- **User degree** reflects activity level (how many games a user owns)

In most real-world systems, degree distributions are **heavy-tailed**, meaning:
- A few items are extremely popular
- Most items have very few interactions

This has strong implications for recommendation bias and model design.

In [ ]:
def degree_distribution(G):
    user_degrees = []
    item_degrees = []

    for node, deg in G.degree():
        if G.nodes[node]['node_type'] == 'user':
            user_degrees.append(deg)
        else:
            item_degrees.append(deg)

    return user_degrees, item_degrees


def plot_degree_distribution(degrees, title):
    counts = Counter(degrees)
    x, y = zip(*sorted(counts.items()))

    plt.figure()
    plt.scatter(x, y)
    plt.xscale('log')
    plt.yscale('log')
    plt.title(title)
    plt.xlabel("Degree")
    plt.ylabel("Frequency")
    plt.show()


user_degrees, item_degrees = degree_distribution(G)
plot_degree_distribution(user_degrees, "User Degree Distribution")
plot_degree_distribution(item_degrees, "Item Degree Distribution")

### Comparison with Random Graph Models

We compare the observed graph to:
- Erdős–Rényi random graph
- Configuration model

This helps determine whether the network structure is random or exhibits real-world properties.

Key insights:
- Real networks typically show **higher clustering** than random graphs
- Similarity to the configuration model indicates that the degree distribution explains much of the structure

This analysis helps validate whether the network follows a **scale-free or heavy-tailed structure**.

In [ ]:
def compare_random_graphs(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()

    # Erdős–Rényi
    p = (2 * m) / (n * (n - 1))
    G_er = nx.erdos_renyi_graph(n, p)

    # Configuration model
    degree_seq = [d for _, d in G.degree()]
    G_conf = nx.configuration_model(degree_seq)
    G_conf = nx.Graph(G_conf)
    G_conf.remove_edges_from(nx.selfloop_edges(G_conf))

    print("Original clustering:", nx.average_clustering(G))
    print("ER clustering:", nx.average_clustering(G_er))
    print("Config clustering:", nx.average_clustering(G_conf))


compare_random_graphs(G)

### Centrality Analysis

We compute centrality measures such as:
- Degree centrality
- PageRank

These identify important nodes in the network:
- Highly central items are typically very popular
- Highly central users are highly active

Centrality can serve as a useful baseline signal in recommendation systems.

In [ ]:
def centrality_analysis(G, sample_size=5000):
    sample_nodes = random.sample(list(G.nodes()), min(sample_size, G.number_of_nodes()))
    subG = G.subgraph(sample_nodes)

    deg_cent = nx.degree_centrality(subG)
    pagerank = nx.pagerank(subG)

    top_deg = sorted(deg_cent.items(), key=lambda x: x[1], reverse=True)[:10]
    top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]

    print("Top degree centrality:", top_deg)
    print("Top PageRank:", top_pr)


centrality_analysis(G)

### Projection Graphs

We construct projected graphs:
- User–User (shared items)
- Item–Item (shared users)

These projections reveal implicit relationships:
- Users with similar preferences
- Items that are frequently co-purchased

The item–item projection is especially important for recommendation, as it captures similarity between products.

In [ ]:
def projection_analysis(G):
    users = [n for n, d in G.nodes(data=True) if d['node_type'] == 'user']
    items = [n for n, d in G.nodes(data=True) if d['node_type'] == 'item']

    user_proj = bipartite.weighted_projected_graph(G, users)
    item_proj = bipartite.weighted_projected_graph(G, items)

    print("User projection nodes:", user_proj.number_of_nodes())
    print("Item projection nodes:", item_proj.number_of_nodes())

    return user_proj, item_proj


user_proj, item_proj = projection_analysis(G)

### Community Detection

We apply community detection on the item–item projection graph.

This helps identify clusters of related items, which may correspond to:
- Game genres
- Player preference groups

Community structure is useful for:
- Understanding the dataset
- Improving recommendations through grouping and similarity

In [ ]:
import networkx.algorithms.community as nx_comm

def community_detection(G_proj):
    communities = nx_comm.louvain_communities(G_proj, seed=42)

    print(f"Detected {len(communities)} communities")

    sizes = [len(c) for c in communities]
    print("Largest communities:", sorted(sizes, reverse=True)[:10])

    return communities

item_communities = community_detection(item_proj)

### Assortativity

We measure degree assortativity to understand whether nodes tend to connect to others with similar degree.

In this context:
- Do highly active users interact with popular items?
- Or is interaction more evenly distributed?

This helps characterize the structure of user behavior and item popularity.

In [ ]:
def assortativity(G):
    assort = nx.degree_assortativity_coefficient(G)
    print(f"Degree assortativity: {assort:.4f}")


assortativity(G)

### Path Analysis

We analyze shortest paths within the largest connected component.

This provides insight into:
- How closely connected the network is
- How efficiently information can propagate

Short average path lengths are typical in real-world networks and beneficial for graph-based learning methods.

In [ ]:
def path_analysis(G):
    largest_cc = max(nx.connected_components(G), key=len)
    subG = G.subgraph(largest_cc)

    print("Avg shortest path:", nx.average_shortest_path_length(subG))


path_analysis(G)

### Summary

The network exhibits key properties of real-world recommendation systems:
- High sparsity
- Heavy-tailed degree distributions
- Presence of highly popular items (hubs)

These characteristics will directly influence the design of the recommendation model, particularly Graph Neural Networks, which rely on graph structure for learning embeddings.

Next steps include implementing baseline models and training GNN-based recommenders.